# Instacart Market Basket Analysis - Data Ingest

This notebook ingests the [Instacart Market Basket Analysis dataset](https://www.kaggle.com/datasets/psparks/instacart-market-basket-analysis) from Kaggle.

## Dataset Overview
The dataset contains transaction data from Instacart including:
- Orders
- Products
- Aisles
- Departments
- Order products (prior and train)

## Prerequisites
- Kaggle API credentials (kaggle.json file with your API key)

In [0]:
%pip install kaggle -q

In [0]:
import os
import json

# Setup Kaggle credentials
# You need to provide your kaggle.json file content
# Download it from: https://www.kaggle.com/settings -> API -> Create New API Token

kaggle_credentials = {
    "username": "username",
    "key": "api_key"
}

# Create .kaggle directory
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

# Write credentials
with open(os.path.join(kaggle_dir, "kaggle.json"), "w") as f:
    json.dump(kaggle_credentials, f)

# Set permissions
os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)

print("✓ Kaggle credentials configured")

In [0]:
import kaggle
from kaggle.api.kaggle_api_extended import KaggleApi

# Initialize Kaggle API
api = KaggleApi()
api.authenticate()

# Create temporary download directory
download_path = "/tmp/instacart_data"
os.makedirs(download_path, exist_ok=True)

# Download the dataset
print("Downloading Instacart dataset from Kaggle...")
api.dataset_download_files(
    'psparks/instacart-market-basket-analysis',
    path=download_path,
    unzip=True
)

print("✓ Dataset downloaded successfully")

In [0]:
import os

# List all downloaded files
download_path = "/tmp/instacart_data"
files = os.listdir(download_path)

print(f"Downloaded {len(files)} files:")
for file in sorted(files):
    file_path = os.path.join(download_path, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"  - {file} ({size_mb:.2f} MB)")

In [0]:
# Define target path using Unity Catalog Volume
target_volume = "/Volumes/big_data/raw/data/instacart/"

# Copy files from temp to Volumes
import shutil
download_path = "/tmp/instacart_data"

for file in os.listdir(download_path):
    source = os.path.join(download_path, file)
    destination = os.path.join(target_volume, file)
    shutil.copy2(source, destination)
    print(f"✓ Copied {file} to {destination}")

print(f"\n✓ All files copied to {target_volume}")

In [0]:
import pandas as pd

# Read one of the CSV files to verify
target_volume = "/Volumes/big_data/raw/data/instacart/"
orders_df = pd.read_csv(target_volume + "orders.csv")

print(f"Orders dataset shape: {orders_df.shape}")
print(f"\nColumns: {list(orders_df.columns)}")
print(f"\nFirst 5 rows:")
display(orders_df.head())